# Continual-learning baseline trên Google Colab

Trước tiên huấn luyện MobileNetV2 bằng `colab_train_mobilenet.ipynb`. Dataset ban đầu phải có đủ các lớp cấu hình. `dataset_dir` ở bước cập nhật phải là thư mục dữ liệu đã gán nhãn, chỉ chứa ảnh train/update (không dùng ảnh test). Classifier incremental chỉ cập nhật SGD head trên feature MobileNetV2 đóng băng; replay cũng chỉ chứa dữ liệu train đã gán nhãn.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
PROJECT_ROOT = Path('/content/drive/MyDrive/waste_classification')
if not (PROJECT_ROOT / 'configs' / 'config_mwcd.yaml').is_file():
    raise FileNotFoundError(f'Copy the project to Google Drive first: {PROJECT_ROOT}')
os.chdir(PROJECT_ROOT)
%pip install -r requirements.txt

In [ ]:
from pathlib import Path
from src.training.common import load_config
from src.training.continual_learning import ContinualLearningManager

config = load_config('configs/config_mwcd.yaml')
data = config['data']
continual = config['continual']
manager = ContinualLearningManager(
    continual['output_dir'], tuple(data['class_names']), int(data['image_size']),
    seed=int(config['project']['seed']),
    replay_images_per_class=int(continual['replay_images_per_class']),
    replay_images_per_update=int(continual['replay_images_per_update']),
    max_old_validation_accuracy_drop=float(continual['max_old_validation_accuracy_drop']),
    minimum_new_validation_accuracy=float(continual['minimum_new_validation_accuracy']),
    test_fraction=float(data['test_fraction']),
    validation_fraction=float(data['validation_fraction']),
)
base_models = sorted(Path(config['models']['mobilenet']['output_dir']).glob('mobilenet_*.keras'))
if not base_models:
    raise FileNotFoundError('Train a MobileNetV2 model first.')
base_model = base_models[-1]
print('Base model:', base_model)
print('Dataset:', data['dataset_dir'])

In [ ]:
# Run once for the initial dataset; fixed test images are snapshotted separately.
if not manager.initialized:
    initial_result = manager.initialize(base_model, data['dataset_dir'], class_sources=data.get('class_sources'))
    print('Initial held-out test metrics:', initial_result['test_metrics'])
else:
    print('Active version:', manager.active_metadata()['version'])

For each later round, set `update_dir` to a **separate labeled update-training dataset**. Do not include the initial held-out test images. The update routine keeps the fixed test snapshot private from training, splits update examples into train/validation where possible, combines training examples with replay, and activates a version only when the configured quality checks pass.

In [ ]:
from src.data.dataset_loader import inspect_dataset

update_dir = Path('data/update-round-2')  # Change to your labeled update dataset.
if update_dir.exists():
    update_report = inspect_dataset(update_dir, data['class_names'], data.get('class_sources'))
    print('Update counts:', update_report.class_counts)
    if not update_report.is_valid:
        raise ValueError('\n'.join(update_report.errors))
    update_records = list(update_report.records)
    result = manager.update(update_records, source='colab_update_round_2')
    print('Accepted:', result['accepted'])
    print('Before fixed test:', result['before_test_metrics'])
    print('After fixed test:', result['after_test_metrics'])
else:
    print('Upload and label an update dataset before running this cell.')